# NFL Margin of Victory Prediction
Reimplementing Warner (2010) on modern data (2015–2024) to evaluate whether Vegas has gotten sharper.

**Setup:** `pip install pandas numpy scikit-learn`

**Data window:**
- Train: 2015–2022
- Test: 2023–2024 (mirrors the paper's 8-year train / 2-year test split)

In [1]:
import pandas as pd
import numpy as np

TRAIN_SEASONS = list(range(2015, 2023))  # 2015–2022
TEST_SEASONS  = list(range(2023, 2025))  # 2023–2024
ALL_SEASONS   = TRAIN_SEASONS + TEST_SEASONS

BASE_STATS = [
    "points_scored", "points_allowed",
    "total_yards", "total_yards_allowed",
    "rush_yards", "rush_yards_allowed",
    "pass_yards", "pass_yards_allowed",
    "turnovers_taken", "turnovers_lost",
]

/Users/virajacharya/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## 1. Game Schedules, Scores, and Spreads

In [2]:
games = pd.read_csv(
    "https://raw.githubusercontent.com/nflverse/nfldata/master/data/games.csv",
    low_memory=False
)

# Regular season only, completed games, within our window
games = games[
    (games["game_type"] == "REG") &
    (games["result"].notna()) &
    (games["season"].isin(ALL_SEASONS))
].copy()

# result = home_score - away_score
# spread_line = Vegas spread (negative = home favored)
print(f"Games loaded: {len(games)} ({games['season'].min()}–{games['season'].max()})")
games[["season", "week", "home_team", "away_team", "home_score", "away_score", "result", "spread_line"]].head()

Games loaded: 2623 (2015–2024)


,season,week,home_team,away_team,home_score,away_score,result,spread_line
4248,2015,1,NE,PIT,28,21,7,7.5
4249,2015,1,BUF,IND,27,14,13,-1.0
4250,2015,1,CHI,GB,23,31,-8,-5.5
4251,2015,1,HOU,KC,20,27,-7,-1.0
4252,2015,1,JAX,CAR,9,20,-11,-3.0


## 2. Play-by-Play Data
Each season is ~50MB — expect a few minutes to load all 10 seasons.

In [3]:
pbp_frames = []
for yr in ALL_SEASONS:
    url = f"https://github.com/nflverse/nflverse-data/releases/download/pbp/play_by_play_{yr}.csv.gz"
    try:
        df = pd.read_csv(url, compression="gzip", low_memory=False)
        pbp_frames.append(df)
        print(f"  {yr}: {len(df):,} plays")
    except Exception as e:
        print(f"  {yr}: FAILED — {e}")

pbp = pd.concat(pbp_frames, ignore_index=True)
print(f"\nTotal plays: {len(pbp):,}")

  2015: 48,122 plays


  2016: 47,651 plays


  2017: 47,245 plays


  2018: 47,109 plays


  2019: 47,260 plays


...

  2021: 49,922 plays


  2022: 49,434 plays


  2023: 49,665 plays


  2024: 49,492 plays



Total plays: 483,605


## 3. Aggregate Play-by-Play Into Per-Game Team Stats
Warner's 10 base stats: points scored/allowed, total/rush/pass yards (offense + defense), turnovers taken/lost.

In [4]:
plays = pbp[pbp["play_type"].isin(["run", "pass", "qb_kneel", "qb_spike"])].copy()

for col in ["yards_gained", "rushing_yards", "passing_yards", "interception", "fumble_lost"]:
    plays[col] = plays[col].fillna(0)

# Offensive stats (per possession team)
off_stats = plays.groupby(["season", "week", "posteam", "game_id"]).agg(
    total_yards=("yards_gained", "sum"),
    rush_yards=("rushing_yards", "sum"),
    pass_yards=("passing_yards", "sum"),
    interceptions_thrown=("interception", "sum"),
    fumbles_lost=("fumble_lost", "sum"),
).reset_index()
off_stats["turnovers_lost"] = off_stats["interceptions_thrown"] + off_stats["fumbles_lost"]

# Defensive stats (yards/turnovers the opponent racked up against you)
def_stats = plays.groupby(["season", "week", "defteam", "game_id"]).agg(
    total_yards_allowed=("yards_gained", "sum"),
    rush_yards_allowed=("rushing_yards", "sum"),
    pass_yards_allowed=("passing_yards", "sum"),
    turnovers_taken_int=("interception", "sum"),
    turnovers_taken_fum=("fumble_lost", "sum"),
).reset_index()
def_stats["turnovers_taken"] = def_stats["turnovers_taken_int"] + def_stats["turnovers_taken_fum"]
def_stats = def_stats.rename(columns={"defteam": "posteam"})
def_stats = def_stats[["season", "week", "posteam", "game_id",
                        "total_yards_allowed", "rush_yards_allowed",
                        "pass_yards_allowed", "turnovers_taken"]]

team_stats = off_stats.merge(def_stats, on=["season", "week", "posteam", "game_id"], how="inner")

# Add points from game-level data
team_stats = team_stats.merge(
    games[["game_id", "home_team", "away_team", "home_score", "away_score"]],
    on="game_id", how="left"
)
team_stats["points_scored"] = np.where(
    team_stats["posteam"] == team_stats["home_team"],
    team_stats["home_score"], team_stats["away_score"]
)
team_stats["points_allowed"] = np.where(
    team_stats["posteam"] == team_stats["home_team"],
    team_stats["away_score"], team_stats["home_score"]
)

team_stats = team_stats[["season", "week", "posteam", "game_id",
                          "home_team", "away_team"] + BASE_STATS]

print(f"Team-game rows: {len(team_stats):,}")
team_stats.head()

Team-game rows: 5,486


,season,week,posteam,game_id,home_team,away_team,points_scored,points_allowed,total_yards,total_yards_allowed,rush_yards,rush_yards_allowed,pass_yards,pass_yards_allowed,turnovers_taken,turnovers_lost
0,2015,1,ARI,2015_01_NO_ARI,ARI,NO,31.0,19.0,427.0,408.0,120.0,54.0,307.0,355.0,1.0,1.0
1,2015,1,ATL,2015_01_PHI_ATL,ATL,PHI,26.0,24.0,395.0,399.0,105.0,63.0,298.0,336.0,2.0,2.0
2,2015,1,BAL,2015_01_BAL_DEN,DEN,BAL,13.0,19.0,173.0,219.0,73.0,69.0,117.0,175.0,1.0,2.0
3,2015,1,BUF,2015_01_IND_BUF,BUF,IND,27.0,14.0,342.0,306.0,147.0,64.0,195.0,243.0,2.0,0.0
4,2015,1,CAR,2015_01_CAR_JAX,JAX,CAR,20.0,9.0,263.0,265.0,105.0,96.0,175.0,183.0,3.0,1.0


## 4. Season-to-Date Averages and 4-Game Streaks
All features are computed from *prior* games only (shifted by 1) to avoid leakage.

In [5]:
team_stats = team_stats.sort_values(["season", "posteam", "week"]).reset_index(drop=True)

# Season-to-date expanding mean
for col in BASE_STATS:
    team_stats[f"{col}_avg"] = (
        team_stats.groupby(["season", "posteam"])[col]
        .transform(lambda x: x.expanding().mean().shift(1))
    )

# 4-game rolling mean (streak)
for col in BASE_STATS:
    team_stats[f"{col}_streak"] = (
        team_stats.groupby(["season", "posteam"])[col]
        .transform(lambda x: x.rolling(4, min_periods=4).mean().shift(1))
    )

# Win percentage (season-to-date and 4-game)
team_stats["win"] = (team_stats["points_scored"] > team_stats["points_allowed"]).astype(int)
team_stats["win_pct"] = (
    team_stats.groupby(["season", "posteam"])["win"]
    .transform(lambda x: x.expanding().mean().shift(1))
)
team_stats["win_pct_streak"] = (
    team_stats.groupby(["season", "posteam"])["win"]
    .transform(lambda x: x.rolling(4, min_periods=4).mean().shift(1))
)

print(f"Rows with valid 4-game streak: {team_stats['points_scored_streak'].notna().sum():,}")

Rows with valid 4-game streak: 4,096


## 5. Computed Strength Rankings
Eigenvalue-based team strength from Warner (2010) Section 2.2.1. Computed using only prior games in the same season (no leakage).

- `S_ij` = total points team i scored against team j so far
- `a_ij = h((S_ij + 1) / (S_ij + S_ji + 2))` where `h(x) = 0.5 + 0.5·sgn(x−0.5)·√|2x−1|`
- Dominant eigenvector of A (via power iteration) = team strength vector

In [6]:
def h_func(x):
    """Keener smoothing: rewards winning, diminishes blowouts."""
    return 0.5 + 0.5 * np.sign(x - 0.5) * np.sqrt(np.abs(2 * x - 1))

def compute_strength(games_df):
    """
    Compute Keener strength ratings for all teams in games_df.
    Returns dict of team -> strength (dominant eigenvector of A).
    """
    teams = sorted(set(games_df["home_team"]) | set(games_df["away_team"]))
    n = len(teams)
    idx = {t: i for i, t in enumerate(teams)}

    # S[i][j] = total points team i scored against team j
    S = np.zeros((n, n))
    for _, row in games_df.iterrows():
        i, j = idx[row["home_team"]], idx[row["away_team"]]
        S[i][j] += row["home_score"]
        S[j][i] += row["away_score"]

    # Build A matrix
    A = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            total = S[i][j] + S[j][i]
            if total > 0:
                A[i][j] = h_func((S[i][j] + 1) / (total + 2))

    # Power iteration for dominant eigenvector
    r = np.ones(n) / n
    for _ in range(200):
        r_new = A @ r
        norm = r_new.sum()
        if norm == 0:
            break
        r_new /= norm
        if np.allclose(r, r_new, atol=1e-10):
            break
        r = r_new

    return dict(zip(teams, r))

# Compute strength per team per week using only prior games
strength_records = []
for season in ALL_SEASONS:
    season_g = games[games["season"] == season].dropna(subset=["home_score", "away_score"])
    for week in sorted(season_g["week"].unique()):
        prior = season_g[season_g["week"] < week]
        if len(prior) < 4:
            continue
        for team, strength in compute_strength(prior).items():
            strength_records.append({
                "season": season, "week": week,
                "posteam": team, "computed_strength": strength
            })

computed_strength_df = pd.DataFrame(strength_records)
print(f"Keener strength records: {len(computed_strength_df):,}")

# Sanity check: top 5 teams in week 10 of 2022
sample = computed_strength_df[(computed_strength_df["season"] == 2022) & (computed_strength_df["week"] == 10)]
print("\nTop 5 teams by Keener strength, 2022 week 10:")
print(sample.sort_values("computed_strength", ascending=False).head().to_string(index=False))

Keener strength records: 5,246

Top 5 teams by Keener strength, 2022 week 10:
 season  week posteam  computed_strength
   2022    10     BAL           0.051263
   2022    10     CIN           0.046033
   2022    10     MIA           0.043219
   2022    10     NYJ           0.042798
   2022    10     BUF           0.042763


## 6. Build Matchup-Level Dataset
Each row is one game: home team features (H_) and away team features (A_) paired with the result and spread.

In [7]:
FEATURE_COLS = (
    [f"{c}_avg"    for c in BASE_STATS] +
    [f"{c}_streak" for c in BASE_STATS] +
    ["win_pct", "win_pct_streak"]
)

home = team_stats[team_stats["posteam"] == team_stats["home_team"]].copy()
away = team_stats[team_stats["posteam"] == team_stats["away_team"]].copy()

home = home.rename(columns={c: f"H_{c}" for c in FEATURE_COLS})
away = away.rename(columns={c: f"A_{c}" for c in FEATURE_COLS})

matchups = home[["game_id", "season", "week"] + [f"H_{c}" for c in FEATURE_COLS]].merge(
    away[["game_id"] + [f"A_{c}" for c in FEATURE_COLS]],
    on="game_id", how="inner"
)
matchups = matchups.merge(
    games[["game_id", "result", "spread_line", "home_team", "away_team"]],
    on="game_id", how="left"
)

# Merge Keener strength for home and away teams
matchups = matchups.merge(
    computed_strength_df.rename(columns={"posteam": "home_team", "computed_strength": "H_computed_strength"}),
    on=["season", "week", "home_team"], how="left"
)
matchups = matchups.merge(
    computed_strength_df.rename(columns={"posteam": "away_team", "computed_strength": "A_computed_strength"}),
    on=["season", "week", "away_team"], how="left"
)

# Drop games where either team lacks enough history (first 4 weeks of each season)
all_feat_cols = [f"H_{c}" for c in FEATURE_COLS] + [f"A_{c}" for c in FEATURE_COLS]
matchups = matchups.dropna(subset=all_feat_cols)

# Column order: identifiers → target/spread → H features → A features
id_cols   = ["season", "week", "home_team", "away_team"]
meta_cols = ["result", "spread_line"]
h_cols    = [f"H_{c}" for c in FEATURE_COLS] + ["H_computed_strength"]
a_cols    = [f"A_{c}" for c in FEATURE_COLS] + ["A_computed_strength"]
matchups  = matchups[["game_id"] + id_cols + meta_cols + h_cols + a_cols]
matchups  = matchups.set_index("game_id").sort_values(["season", "week"])

print(f"Matchup rows: {len(matchups):,}")
print(f"Columns: {len(id_cols)} identifiers  |  {len(meta_cols)} target/spread  |  {len(h_cols)} H features  |  {len(a_cols)} A features  =  {len(h_cols+a_cols)} total features")
matchups.head()

Matchup rows: 1,885
Columns: 4 identifiers  |  2 target/spread  |  23 H features  |  23 A features  =  46 total features


,season,week,home_team,away_team,result,spread_line,H_points_scored_avg,H_points_allowed_avg,H_total_yards_avg,H_total_yards_allowed_avg,...,A_total_yards_allowed_streak,A_rush_yards_streak,A_rush_yards_allowed_streak,A_pass_yards_streak,A_pass_yards_allowed_streak,A_turnovers_taken_streak,A_turnovers_lost_streak,A_win_pct,A_win_pct_streak,A_computed_strength
game_id,,,,,,,,,,,,,,,,,,,,,
2015_05_WAS_ATL,2015,5,ATL,WAS,6,7.5,34.25,23.25,403.75,390.50,...,288.00,139.50,78.0,251.25,231.25,0.75,1.75,0.50,0.50,0.037714
2015_05_CLE_BAL,2015,5,BAL,CLE,-3,6.0,23.25,26.00,355.00,347.00,...,406.25,89.75,141.5,275.00,277.00,1.50,1.50,0.25,0.25,0.019095
2015_05_SEA_CIN,2015,5,CIN,SEA,3,3.0,30.25,19.25,422.50,364.75,...,279.25,128.00,88.5,244.75,203.00,1.25,1.25,0.50,0.50,0.028690
2015_05_ARI_DET,2015,5,DET,ARI,-25,-4.5,16.50,24.00,292.75,383.00,...,306.75,121.75,107.5,288.75,208.50,1.75,1.25,0.75,0.75,0.038209
2015_05_IND_HOU,2015,5,HOU,IND,-7,4.5,19.25,27.00,384.75,344.00,...,387.50,87.50,119.5,258.75,276.00,0.75,2.75,0.50,0.50,0.019116


## 6. Baseline: Linear Regression vs. Vegas Spread
A sanity check before swapping in the Gaussian Process.

In [8]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

X_cols = [f"H_{c}" for c in FEATURE_COLS] + [f"A_{c}" for c in FEATURE_COLS]

train = matchups[matchups["season"].isin(TRAIN_SEASONS)].dropna(subset=X_cols + ["result"])
test  = matchups[matchups["season"].isin(TEST_SEASONS)].dropna(subset=X_cols + ["result", "spread_line"])

model = LinearRegression().fit(train[X_cols], train["result"])
test = test.copy()
test["pred"] = model.predict(test[X_cols])

mae_model  = mean_absolute_error(test["result"], test["pred"])
mae_spread = mean_absolute_error(test["result"], -test["spread_line"])  # nflverse spread_line is away-team perspective

acc_model  = (np.sign(test["pred"]) == np.sign(test["result"])).mean()
acc_spread = (np.sign(-test["spread_line"]) == np.sign(test["result"])).mean()

print(f"Test set: {len(test)} games ({TEST_SEASONS[0]}–{TEST_SEASONS[-1]})\n")
print(f"{'Metric':<28} {'Linear Reg':>12} {'Vegas Spread':>12}")
print("-" * 54)
print(f"{'MAE (margin of victory)':<28} {mae_model:>12.2f} {mae_spread:>12.2f}")
print(f"{'Winner accuracy':<28} {acc_model:>11.1%} {acc_spread:>11.1%}")

Test set: 416 games (2023–2024)

Metric                         Linear Reg Vegas Spread
------------------------------------------------------
MAE (margin of victory)             10.28        14.49
Winner accuracy                    63.9%       27.9%


## Next Steps

1. **Gaussian Process model** — replace `LinearRegression` with `GaussianProcessRegressor` using an RBF + WhiteKernel (matches the paper's squared-exponential + noise kernel).
2. **Feature selection** — forward search from Section 4.1 of the paper.
3. **Keener rankings** — eigenvalue-based team strength from Section 2.2.1.
4. **Betting framework** — use GP posterior uncertainty to flag bets per Section 4.2.
5. **Era comparison** — run the same pipeline on the paper's original 2000–2009 window and compare MAE/accuracy to our 2015–2024 results.